In [59]:
import numpy as np
import pandas as pd
from scipy import stats

# Load Dataset
sales_df = pd.read_csv("/messy_sales_data.csv")
print(f"Initial Row Count: {len(sales_df)}")

Initial Row Count: 159


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [60]:
# Initial Data Snapshot
def initial_copy(df):
    # Calculate the total number of rows in the DataFrame
    initial_row_count = len(df)

    # Calculate the total number of missing/null values across all columns
    initial_null_count = df.isnull().sum().sum()

    # Calculate the total number of exact duplicate rows
    initial_dup_count = df.duplicated().sum()

    # Print a summary showing the baseline metrics of the dataset
    print(
        f"Initial Row Count: {initial_row_count}\n"
        f"Initial Null Count: {initial_null_count}\n"
        f"Initial Duplicate Count: {initial_dup_count}"
    )

    # Return a deep copy of the original DataFrame.
    # This prevents changes to df from affecting the baseline backup copy.
    return df.copy()

In [61]:
# Column Name Standardisation
def col_standard(df):
    # Clean the column names by:
    # 1. str.strip() - Removing leading and trailing whitespaces
    # 2. str.lower() - Converting all characters to lowercase
    # 3. str.replace() - Replacing spaces between words with underscores
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # Print the newly formatted column names to the console
    display(df.columns)

    # Return the updated DataFrame
    return df


In [62]:
def generate_quality_report(data):
    # Data Quality Report Function
    print("--- DATA QUALITY REPORT ---")
    print("\nNull Counts per Column:")
    print(data.isnull().sum())
    print(f"\nDuplicate Rows: {data.duplicated().sum()}")
    print("\nData Types:")
    print(data.dtypes)
    print("\nBasic Numeric Summary (Value Range Anomalies Check):")
    print(data.describe())

In [63]:
def find_partial_duplicates(df, primary_key_col):
    """
    Dynamically identifies partial duplicates based on a unique identifier.
    Returns a dataframe containing only the conflicting records, sorted by the ID.
    """
    # 1. Clean column strings to prevent hidden space issues
    df_temp = df.copy()

    # 2. Get all rows that share a duplicated primary key
    all_dup_ids = df_temp[df_temp.duplicated(subset=[primary_key_col], keep=False)]

    # 3. Filter out exact duplicates (if entire rows match, they aren't 'partial' conflicts)
    # This leaves ONLY rows with the same ID but variations in other columns
    partial_dups = all_dup_ids.drop_duplicates(keep=False)

    # 4. Sort by the primary key so conflicts sit directly next to each other
    dups = partial_dups.sort_values(by=primary_key_col)

    print("Partial Duplicates:\n")
    display(dups)
    return dups

In [64]:
def resolve_by_latest_date(df, primary_key, date_col):
    """Resolves conflicts by keeping the row with the most recent date."""
    df_temp = df.copy()

    # 1. Safely convert to datetime (coercing unparseable text to NaT)
    df_temp['_parsed_date'] = pd.to_datetime(df_temp[date_col], errors='coerce')

    # 2. Sort so that valid, newer dates flow to the bottom
    df_temp = df_temp.sort_values(by=[primary_key, '_parsed_date'], ascending=[True, True])

    # 3. Drop duplicates keeping the LAST entry (the most recent)
    cleaned_df = df_temp.drop_duplicates(subset=[primary_key], keep='last')

    # 4. Drop the temporary helper column
    clean_ = cleaned_df.drop(columns=['_parsed_date'])
    display(clean_)
    return clean_

In [65]:
# Duplicate Removal
def remove_dup(df):
    # Calculate the total number of duplicate rows before removal
    duplicates_before = df.duplicated().sum()

    # Drop duplicate rows, keeping only the first occurrence of each duplicate
    df = df.drop_duplicates(keep="first")

    # The number of duplicates removed matches the count found before dropping
    duplicates_removed = duplicates_before

    # Print a summary statement showing how many duplicates were deleted
    print(f"\nDuplicates removed: {duplicates_removed}")
    display (df.head())
    # Return the cleaned DataFrame
    return df

In [66]:
# Define a function to standardise a text column by extracting numbers and converting it to a numeric type
def text_col_standard(df, col):
    """
    Cleans a dataframe column by stripping out all non-numeric characters
    and converting the remaining digits into actual numbers.

    Parameters:
    df  (DataFrame): The pandas DataFrame to modify.
    col (str):       The name of the column to clean (e.g., 'price' or 'id').

    Returns:
    DataFrame: The modified DataFrame with the cleaned, numeric column.
    """
    # Step 1: Remove all characters that are NOT digits (0-9)
    # The regex '[^0-9.]' matches letters, symbols (like $), spaces, replacing them with nothing ''
    df.loc[:,col] = df[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)

    # Step 2: Convert the cleaned text strings into actual mathematical numbers (integers/floats)
    df.loc[:,col] = pd.to_numeric(df[col])

    display (df.head())
    # Step 3: Return the updated DataFrame
    return df

In [67]:
# Date standardisation (e.g., date_column)
def date_standard(df, date_col):
    df = df.copy()
    # Check if the specified date column exists in the DataFrame
    if date_col in df.columns:
        # Convert the column to datetime format
        # "mixed" handles multiple date formats in the same column
        # "coerce" turns invalid date formats into NaT (Not a Time) values
        df[date_col] = pd.to_datetime(
            df[date_col], format="mixed", dayfirst=False, errors="coerce"
        )
        print("Standardised Dates:\n")
        display(df[date_col])
        # Return the updated DataFrame
    return df

In [68]:
# Missing Data Handling & Justification Strategy
def fill_nulls(df):
    df = df.copy()
    # --- Strategy 1: Numeric columns -> Mean Imputation ---
    # Find all numeric columns (integers and floats) in the DataFrame
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        # Check if the numeric column contains any missing values
        if df[col].isnull().sum() > 0:
            mean_value = df[col].mean()

            # Fill missing values with the mean (robust to outliers/skewness)
            df[col] = df[col].fillna(mean_value)
            print(f"Numeric Column: {col} \nImputation (Median): {mean_value}")

    # --- Strategy 2: Categorical columns -> Mode Imputation ---
    # Find all categorical columns (strings, objects, and categories)
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        # Check if the categorical column contains any missing values
        if df[col].isnull().sum() > 0:
            # Check if a mode exists (handles completely empty columns safely)
            if not df[col].mode().empty:
                mode_value = df[col].mode()[0]

                # Fill missing values with the most frequent category
                df[col] = df[col].fillna(mode_value)
                print(
                    f"Non-numeric Column: {col} \nImputation (Mode): {mode_value}"
                )

    # Return the clean DataFrame with imputed values
    date_cols = df.select_dtypes(include =[np.datetime64]).columns
    for col in date_cols:
        if df[col].isnull().sum()>0:
            if not df[col].mode().empty:
                mode_value = df[col].mode()[0]
                df[col] = df[col].fillna(mode_value)
                print(
                    f"Date Column: {col} \nImputation (Mode): {mode_value}"
                )
    display(df)
    return df


In [70]:
def outlier_treat(df, col_name):
    # Create an independent copy to prevent setting warnings
    df = df.copy()

    # Strip any remaining whitespace and force column to float
    df[col_name] = df[col_name].astype(str).str.strip()
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')

    # Drop rows where the conversion failed (became NaN)
    df = df.dropna(subset=[col_name])

    # Calculate absolute Z-scores safely on numerical data
    z_scores = np.abs(stats.zscore(df[col_name]))

    # Return rows that fall within 3 standard deviations
    clean_df = df[z_scores < 3]
    print("Clean Dataframe:\n")
    display (clean_df)
    return clean_df



In [71]:
def _save(df, file_path):
    df.to_csv(file_path, index=include_index)

In [72]:
initial_copy(sales_df)

Initial Row Count: 159
Initial Null Count: 22
Initial Duplicate Count: 8


,Order ID,purchase_date,CustomerName,Category,Quantity,Unit_Price_$,Delivery City
0,1028,2025-01-02 04:00:00,Customer_28,ELECTRONICS,4.0,259.2,Houston
1,1083,2025-01-04 11:00:00,Customer_83,Clothing,4.0,126.44,Los Angeles
2,1101,2025-01-05 05:00:00,Customer_101,Books,2.0,348.56,Chicago
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
4,1054,2025-01-03 06:00:00,Customer_54,Clothing,3.0,$260.27,Los Angeles
...,...,...,...,...,...,...,...
154,1007,2025-01-01 07:00:00,Customer_7,ELECTRONICS,1.0,172.14,New York
155,1050,2025-01-03 02:00:00,Customer_50,Books,3.0,219.28,New York
156,1108,2025-01-05 12:00:00,Customer_108,Home & Kitchen,2.0,$439.9,NY
157,1037,2025-01-02 13:00:00,Customer_37,Electronics,1.0,465.55,Los Angeles


In [73]:
generate_quality_report(sales_df)

--- DATA QUALITY REPORT ---

Null Counts per Column:
 Order ID         0
purchase_date     0
CustomerName     13
Category          0
Quantity          9
 Unit_Price_$     0
Delivery City     0
dtype: int64

Duplicate Rows: 8

Data Types:
 Order ID          int64
purchase_date     object
CustomerName      object
Category          object
Quantity         float64
 Unit_Price_$     object
Delivery City     object
dtype: object

Basic Numeric Summary (Value Range Anomalies Check):
         Order ID     Quantity
count   159.000000  150.000000
mean   1074.113208    9.226667
std      43.348745   81.367354
min    1000.000000   -5.000000
25%    1036.500000    2.000000
50%    1074.000000    3.000000
75%    1111.500000    4.000000
max    1149.000000  999.000000


In [74]:
standard_cols = col_standard(sales_df)

Index(['order_id', 'purchase_date', 'customername', 'category', 'quantity',
       'unit_price_$', 'delivery_city'],
      dtype='object')

In [75]:
dup_free = remove_dup(standard_cols)


Duplicates removed: 8


,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
0,1028,2025-01-02 04:00:00,Customer_28,ELECTRONICS,4.0,259.2,Houston
1,1083,2025-01-04 11:00:00,Customer_83,Clothing,4.0,126.44,Los Angeles
2,1101,2025-01-05 05:00:00,Customer_101,Books,2.0,348.56,Chicago
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
4,1054,2025-01-03 06:00:00,Customer_54,Clothing,3.0,$260.27,Los Angeles


In [76]:
text_col_clean = text_col_standard(dup_free, "unit_price_$")

,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
0,1028,2025-01-02 04:00:00,Customer_28,ELECTRONICS,4.0,259.2,Houston
1,1083,2025-01-04 11:00:00,Customer_83,Clothing,4.0,126.44,Los Angeles
2,1101,2025-01-05 05:00:00,Customer_101,Books,2.0,348.56,Chicago
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
4,1054,2025-01-03 06:00:00,Customer_54,Clothing,3.0,260.27,Los Angeles


In [77]:
date_clean = date_standard(dup_free,"purchase_date")

Standardised Dates:



,purchase_date
0,2025-01-02 04:00:00
1,2025-01-04 11:00:00
2,2025-01-05 05:00:00
3,2025-01-01 10:00:00
4,2025-01-03 06:00:00
...,...
153,2025-01-02 02:00:00
154,2025-01-01 07:00:00
155,2025-01-03 02:00:00
156,2025-01-05 12:00:00


In [78]:
null_free = fill_nulls(date_clean)

Numeric Column: quantity 
Imputation (Median): 9.591549295774648
Non-numeric Column: customername 
Imputation (Mode): Customer_10
Date Column: purchase_date 
Imputation (Mode): 2025-01-01 00:00:00


,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
0,1028,2025-01-02 04:00:00,Customer_28,ELECTRONICS,4.0,259.2,Houston
1,1083,2025-01-04 11:00:00,Customer_83,Clothing,4.0,126.44,Los Angeles
2,1101,2025-01-05 05:00:00,Customer_101,Books,2.0,348.56,Chicago
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
4,1054,2025-01-03 06:00:00,Customer_54,Clothing,3.0,260.27,Los Angeles
...,...,...,...,...,...,...,...
153,1026,2025-01-02 02:00:00,Customer_10,Clothing,2.0,321.84,Houston
154,1007,2025-01-01 07:00:00,Customer_7,ELECTRONICS,1.0,172.14,New York
155,1050,2025-01-03 02:00:00,Customer_50,Books,3.0,219.28,New York
156,1108,2025-01-05 12:00:00,Customer_108,Home & Kitchen,2.0,439.9,NY


In [79]:
outlier_free = outlier_treat(null_free, "unit_price_$")

Clean Dataframe:



,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
0,1028,2025-01-02 04:00:00,Customer_28,ELECTRONICS,4.0,259.20,Houston
1,1083,2025-01-04 11:00:00,Customer_83,Clothing,4.0,126.44,Los Angeles
2,1101,2025-01-05 05:00:00,Customer_101,Books,2.0,348.56,Chicago
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
4,1054,2025-01-03 06:00:00,Customer_54,Clothing,3.0,260.27,Los Angeles
...,...,...,...,...,...,...,...
153,1026,2025-01-02 02:00:00,Customer_10,Clothing,2.0,321.84,Houston
154,1007,2025-01-01 07:00:00,Customer_7,ELECTRONICS,1.0,172.14,New York
155,1050,2025-01-03 02:00:00,Customer_50,Books,3.0,219.28,New York
156,1108,2025-01-05 12:00:00,Customer_108,Home & Kitchen,2.0,439.90,NY


In [80]:
find_partial_duplicates(outlier_free,"order_id")

Partial Duplicates:



,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
31,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,Chicago


,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
3,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,New York
31,1010,2025-01-01 10:00:00,Customer_10,Electronics,3.0,169.34,Chicago


In [81]:
final_df = resolve_by_latest_date(outlier_free,"order_id","purchase_date")

,order_id,purchase_date,customername,category,quantity,unit_price_$,delivery_city
32,1000,2025-01-01 00:00:00,Customer_10,ELECTRONICS,9.591549,367.21,L.A.
108,1001,2025-01-01 01:00:00,Customer_1,Electronics,4.000000,387.92,New York
16,1002,2025-01-01 02:00:00,Customer_2,Home & Kitchen,1.000000,46.28,Chicago
144,1003,2025-01-01 03:00:00,Customer_3,Clothing,3.000000,185.65,New York
56,1004,2025-01-01 04:00:00,Customer_4,Clothing,3.000000,66.78,Los Angeles
...,...,...,...,...,...,...,...
101,1145,2025-01-07 01:00:00,Customer_145,Home & Kitchen,3.000000,332.23,Los Angeles
13,1146,2025-01-07 02:00:00,Customer_146,Books,3.000000,288.47,Houston
8,1147,2025-01-07 03:00:00,Customer_147,BOOKS,3.000000,55.90,Houston
39,1148,2025-01-07 04:00:00,Customer_148,Electronics,1.000000,190.18,New York


In [ ]:
!apt install python3-wordcloud --yes

Installing:                     
  python3-wordcloud

Installing dependencies:
  fonts-noto-mono

Summary:
  Upgrading: 0, Installing: 2, Removing: 0, Not Upgrading: 0
  Download size: 520 kB
  Space needed: 1483 kB / 26.8 GB available

Get:1 http://ports.ubuntu.com/ubuntu-ports resolute/main arm64 fonts-noto-mono all 20201225-2build1 [434 kB]
Get:2 http://ports.ubuntu.com/ubuntu-ports resolute/universe arm64 python3-wordcloud arm64 1.9.4+dfsg-1build1 [86.5 kB]
Fetched 520 kB in 13s (38.6 kB/s)                                              

78Selecting previously unselected package fonts-noto-mono.
(Reading database ... 79359 files and directories currently installed.)
Preparing to unpack .../fonts-noto-mono_20201225-2build1_all.deb ...
7Progress: [  0%] [                                                          ] 87Progress: [ 11%] [██████▍                                                   ] 8Unpacking fonts-noto-mono (20201225-2build1) ...
7Progress: [ 22%] [████████████▉     